# SDRF Extraction Pipeline — PRIDE API + Regex + BioBERT QA
**Three-layer architecture:**
1. **PRIDE API** — fetches structured metadata already registered by the researchers
2. **Regex** — extracts protocol parameters from paper text (instrument, tolerances, label, cleavage)
3. **BioBERT QA** — extractive question-answering for hard columns regex misses (disease, tissue, cell type)

Runs fully offline on Kaggle T4. `dmis-lab/biobert-large-cased-v1.1-squad` must be attached as a dataset
OR internet must be enabled for the first run (model is ~1.3 GB).

In [4]:
import os, re, json, time, difflib
from collections import defaultdict, Counter
from pathlib import Path

import requests
import pandas as pd
import torch
from transformers import pipeline as hf_pipeline
from tqdm import tqdm

PRIDE_TIMEOUT = 15
PX_TIMEOUT    = 15

# ── Paths — works on both Kaggle and locally ──────────────────────────────
IS_KAGGLE   = Path('/kaggle').exists()
BASE_PATH   = Path('/kaggle/input/harmonizing-the-data-of-your-data') if IS_KAGGLE \
              else Path.cwd().parent / 'data'

TRAIN_SDRF_DIR = BASE_PATH / 'TrainingSDRFs'
SAMPLE_SUB     = BASE_PATH / 'SampleSubmission.csv'
OUT_PATH       = Path('/kaggle/working/submission.csv') if IS_KAGGLE \
                 else Path.cwd().parent / 'outputs' / 'submission_pride_regex_biobert.csv'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Test PubText — probe several known path variants
_pubtext_candidates = [
    BASE_PATH / 'Test_PubText' / 'Test PubText',
    BASE_PATH / 'Test_PubText',
    BASE_PATH / 'TestPubText',
    BASE_PATH / 'Test PubText',
    BASE_PATH.parent / 'data' / 'TestPubText',
]
TEST_TEXT_DIR = next((p for p in _pubtext_candidates if p.exists()), _pubtext_candidates[0])

# BioBERT QA model — attach dataset or use HF hub
BIOBERT_MODEL = '/kaggle/input/biobert-squad/biobert-large-squad' \
    if IS_KAGGLE and Path('/kaggle/input/biobert-squad').exists() \
    else 'dmis-lab/biobert-large-cased-v1.1-squad'

device = 0 if torch.cuda.is_available() else -1
print(f'Device     : {"GPU" if device==0 else "CPU"}')
print(f'Base path  : {BASE_PATH}')
print(f'PubText    : {TEST_TEXT_DIR} — exists: {TEST_TEXT_DIR.exists()}')
print(f'BioBERT    : {BIOBERT_MODEL}')

Device     : GPU
Base path  : c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data
PubText    : c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\TestPubText — exists: True
BioBERT    : dmis-lab/biobert-large-cased-v1.1-squad


## 1. Load training data — build vocabulary and global modes

In [6]:
sample_sub  = pd.read_csv(SAMPLE_SUB)
id_cols     = ['ID', 'PXD', 'Raw Data File', 'Usage']
target_cols = [c for c in sample_sub.columns if c not in id_cols and 'Unnamed' not in c]
base_tgt    = sorted(set([re.sub(r'\.\d+$', '', c) for c in target_cols]))
all_base    = set(base_tgt)

def strip_wrapper(col):
    """'Characteristics[Organism]' → 'Organism', 'Comment[Instrument]' → 'Instrument'"""
    m = re.match(r'(?:characteristics|comment|factor value)\[(.+?)\]', col, re.I)
    return m.group(1) if m else col

def find_col_in_df(col, df_cols):
    """Try exact match, then base (no suffix), then stripped wrapper."""
    if col in df_cols: return col
    base = re.sub(r'\.\d+$', '', col)
    if base in df_cols: return base
    stripped = strip_wrapper(base)
    if stripped in df_cols: return stripped
    return None

col_counters = {col: Counter() for col in target_cols}
col_vocab    = defaultdict(set)
train_files  = sorted(TRAIN_SDRF_DIR.glob('*_cleaned.sdrf.tsv')) if TRAIN_SDRF_DIR.exists() else []

for fp in train_files:
    df = pd.read_csv(fp, sep='\t', low_memory=False)
    df_cols = set(df.columns)
    for col in target_cols:
        col_match = find_col_in_df(col, df_cols)
        if col_match:
            vals = df[col_match].dropna().astype(str)
            vals = vals[~vals.isin(['Not Applicable', 'not applicable', 'NA'])]
            col_counters[col].update(vals.tolist())
            col_vocab[re.sub(r'\.\d+$', '', col)].update(vals.tolist())

global_modes = {}
non_na_ratio = {}
n_train = max(len(train_files), 1)
for col in target_cols:
    total = sum(col_counters[col].values())
    if total > 0:
        global_modes[col] = col_counters[col].most_common(1)[0][0]
        non_na_ratio[col] = total / n_train
    else:
        global_modes[col] = 'Not Applicable'
        non_na_ratio[col] = 0.0

# Per-PXD lookup from training SDRFs (if a test PXD appeared in training)
train_pxd_sdrf = {}
for fp in train_files:
    pxd = fp.stem.split('_')[0]
    df  = pd.read_csv(fp, sep='\t', low_memory=False)
    df_cols = set(df.columns)
    pxd_vals = {}
    for col in target_cols:
        col_match = find_col_in_df(col, df_cols)
        if col_match:
            vals = df[col_match].dropna().astype(str)
            vals = vals[~vals.str.lower().isin(['not applicable', 'n/a', 'na', ''])]
            uniq = list(vals.unique())
            if uniq:
                pxd_vals[col] = uniq
    train_pxd_sdrf[pxd] = pxd_vals

print(f'Training SDRFs   : {len(train_files)}')
print(f'Target columns   : {len(target_cols)}')
print(f'Vocabulary size  : {sum(len(v) for v in col_vocab.values()):,} terms')
print(f'Top global modes :')
for col in list(global_modes)[:5]:
    print(f'  {col}: {global_modes[col]!r}')


Training SDRFs   : 103
Target columns   : 77
Vocabulary size  : 1,538 terms
Top global modes :
  Characteristics[Age]: 'not available'
  Characteristics[AlkylationReagent]: 'IAA'
  Characteristics[AnatomicSiteTumor]: 'Not Applicable'
  Characteristics[AncestryCategory]: 'not available'
  Characteristics[BMI]: '26.5'


## 2. Normalisation dictionaries

In [7]:
organism_norm = {
    'human': '9606 (Homo sapiens)', 'homo sapiens': '9606 (Homo sapiens)',
    'mouse': '10090 (Mus musculus)', 'mice': '10090 (Mus musculus)',
    'murine': '10090 (Mus musculus)', 'mus musculus': '10090 (Mus musculus)',
    'rat': '10116 (Rattus norvegicus)', 'rattus norvegicus': '10116 (Rattus norvegicus)',
    'yeast': '4932 (Saccharomyces cerevisiae)',
    'saccharomyces cerevisiae': '4932 (Saccharomyces cerevisiae)',
    'e. coli': '562 (Escherichia coli)', 'escherichia coli': '562 (Escherichia coli)',
    'drosophila melanogaster': '7227 (Drosophila melanogaster)',
    'zebrafish': '7955 (Danio rerio)', 'danio rerio': '7955 (Danio rerio)',
    'arabidopsis thaliana': '3702 (Arabidopsis thaliana)',
    'pig': '9823 (Sus scrofa)', 'porcine': '9823 (Sus scrofa)',
    'bovine': '9913 (Bos taurus)', 'bos taurus': '9913 (Bos taurus)',
    'chicken': '9031 (Gallus gallus)', 'gallus gallus': '9031 (Gallus gallus)',
    'c. elegans': '6239 (Caenorhabditis elegans)',
    'caenorhabditis elegans': '6239 (Caenorhabditis elegans)',
}

tissue_norm = {
    'brain': 'NT=brain;AC=UBERON:0000955',
    'liver': 'NT=liver;AC=UBERON:0002107',
    'lung': 'NT=lung;AC=UBERON:0002048',
    'heart': 'NT=heart;AC=UBERON:0000948',
    'kidney': 'NT=kidney;AC=UBERON:0002113',
    'colon': 'NT=colon;AC=UBERON:0001155',
    'breast': 'NT=breast;AC=UBERON:0000310',
    'prostate': 'NT=prostate gland;AC=UBERON:0002367',
    'pancreas': 'NT=pancreas;AC=UBERON:0001264',
    'plasma': 'NT=blood plasma;AC=UBERON:0001969',
    'blood plasma': 'NT=blood plasma;AC=UBERON:0001969',
    'serum': 'NT=blood serum;AC=UBERON:0001977',
    'blood': 'NT=blood;AC=UBERON:0000178',
    'urine': 'NT=urine;AC=UBERON:0001088',
    'cerebrospinal fluid': 'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'csf': 'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'bone marrow': 'NT=bone marrow;AC=UBERON:0002371',
    'spleen': 'NT=spleen;AC=UBERON:0002106',
    'muscle': 'NT=skeletal muscle;AC=UBERON:0001134',
    'thymus': 'NT=thymus;AC=UBERON:0002370',
    'lymph node': 'NT=lymph node;AC=UBERON:0000029',
    'adipose': 'NT=adipose tissue;AC=UBERON:0001013',
    'skin': 'NT=skin of body;AC=UBERON:0002097',
    'hippocampus': 'NT=hippocampal formation;AC=UBERON:0002421',
    'cortex': 'NT=cerebral cortex;AC=UBERON:0000956',
    'cerebellum': 'NT=cerebellum;AC=UBERON:0002037',
    'striatum': 'NT=striatum;AC=UBERON:0002435',
    'substantia nigra': 'NT=substantia nigra;AC=UBERON:0002038',
    'pbmc': 'NT=peripheral blood mononuclear cell;AC=CL:0000057',
    'testis': 'NT=testis;AC=UBERON:0000473',
    'ovary': 'NT=ovary;AC=UBERON:0000992',
    'stomach': 'NT=stomach;AC=UBERON:0000945',
    'retina': 'NT=retina;AC=UBERON:0000966',
    'thyroid': 'NT=thyroid gland;AC=UBERON:0002046',
    'endometrium': 'NT=endometrium;AC=UBERON:0001295',
    'placenta': 'NT=placenta;AC=UBERON:0001987',
}

instrument_norm = {
    'q exactive hf-x':        'AC=MS:1003027;NT=Q Exactive HF-X',
    'q exactive hf':          'AC=MS:1002523;NT=Q Exactive HF',
    'q exactive plus':        'AC=MS:1002634;NT=Q Exactive Plus',
    'q-exactive plus':        'AC=MS:1002634;NT=Q Exactive Plus',
    'q exactive':             'AC=MS:1001911;NT=Q Exactive',
    'orbitrap astral':        'AC=MS:1003378;NT=Orbitrap Astral',
    'orbitrap fusion lumos':  'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
    'orbitrap fusion':        'AC=MS:1002416;NT=Orbitrap Fusion',
    'orbitrap eclipse':       'AC=MS:1003029;NT=Orbitrap Eclipse',
    'exploris 480':           'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'orbitrap exploris 480':  'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'ltq orbitrap velos':     'AC=MS:1001742;NT=LTQ Orbitrap Velos',
    'ltq orbitrap elite':     'AC=MS:1001910;NT=LTQ Orbitrap Elite',
    'ltq orbitrap xl':        'AC=MS:1000556;NT=LTQ Orbitrap XL',
    'ltq orbitrap':           'AC=MS:1000449;NT=LTQ Orbitrap',
    'timstof pro':            'AC=MS:1003231;NT=timsTOF Pro',
    'timstof':                'AC=MS:1002817;NT=timsTOF',
    'triple tof 6600':        'AC=MS:1000931;NT=TripleTOF 6600',
    'triple tof 5600':        'AC=MS:1000931;NT=TripleTOF 5600',
    'triple tof':             'AC=MS:1000931;NT=TripleTOF 6600',
}

CELL_LINES = [
    'HEK293T','HEK293','HEK-293','HeLa','U2OS','MCF7','MCF-7','A549','Jurkat',
    'K562','HCT116','HepG2','CHO','PC3','LNCaP','THP-1','SH-SY5Y','Caco-2',
    'NIH3T3','RAW264.7','U87','U251','T47D','MDA-MB-231','MDA-MB-468','PANC-1',
    'HL-60','Ramos','NCI-H1299','NCI-H460','SW480','SW620','LoVo','HT-29','BV2',
    'Vero','HUVEC','B16','C2C12','3T3-L1','U937','RPMI 8226','MEF','iPSC',
    'DLD-1','RKO','Huh7','PC-9','H1975','A375','SKBR3','BT474','ZR-75-1',
]

def fmt_label(n):
    n = str(n).lower().strip()
    if any(x in n for x in ['label free', 'label-free', 'lfq']):
        return 'AC=MS:1002038;NT=label free sample'
    if 'tmt' in n:
        m = re.search(r'tmt[\s\-]?(\d+)', n)
        p = m.group(1) if m else '6'
        acc = {'2':'MS:1002456','6':'MS:1002453','10':'MS:1002454',
               '11':'MS:1002454','16':'MS:1003998','18':'MS:1003999'}
        return f'AC={acc.get(p,"MS:1002453")};NT=TMT{p}plex'
    if 'itraq' in n:
        m = re.search(r'itraq[\s\-]?(\d+)', n)
        p = m.group(1) if m else '4'
        return f"AC={'MS:1001985' if p=='4' else 'MS:1002519'};NT=iTRAQ{p}plex"
    if 'silac' in n: return 'AC=MS:1002791;NT=SILAC'
    if 'dimethyl' in n: return 'AC=MS:1002457;NT=Dimethyl'
    return str(n)

def fmt_instrument(raw):
    n = raw.lower().strip()
    for key, val in instrument_norm.items():
        if key in n:
            return val
    return raw

def fuzzy_snap(value, base_col, cutoff=0.82):
    if not value or base_col not in col_vocab:
        return value
    matches = difflib.get_close_matches(value, list(col_vocab[base_col]), n=1, cutoff=cutoff)
    return matches[0] if matches else value

print('Normalisation dictionaries ready.')

Normalisation dictionaries ready.


## 3. Load test PubText JSONs

In [8]:
SECTIONS = ['TITLE','ABSTRACT','METHODS','MATERIALS AND METHODS',
            'EXPERIMENTAL','SAMPLE PREPARATION','MASS SPECTROMETRY',
            'LC-MS','PROTEIN DIGESTION','DATA ACQUISITION','CELL CULTURE']
METHOD_KWS = ['method','material','protocol','digest','spectr','chromat',
              'prep','enrichment','culture','experimental','lc','ms']

def load_pub_dict(path):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8', errors='replace'))
    except Exception:
        return {}

def get_text(pub_dict, max_chars=None):
    """Concatenate sections in priority order: methods first, then abstract/title."""
    parts = []
    for key in SECTIONS:
        val = pub_dict.get(key, '')
        if isinstance(val, list): val = ' '.join(str(x) for x in val)
        if val.strip(): parts.append(val.strip())
    for key, val in pub_dict.items():
        if key.upper() in SECTIONS: continue
        if any(kw in key.lower() for kw in METHOD_KWS):
            if isinstance(val, list): val = ' '.join(str(x) for x in val)
            if val.strip(): parts.append(val.strip())
    for key in ['ABSTRACT', 'TITLE']:
        val = pub_dict.get(key, '')
        if isinstance(val, list): val = ' '.join(str(x) for x in val)
        if val.strip(): parts.append(val.strip())
    text = ' '.join(parts)
    return text[:max_chars] if max_chars else text

test_docs = {}
if TEST_TEXT_DIR.exists():
    for fp in sorted(TEST_TEXT_DIR.glob('*.json')):
        pxd = fp.stem.split('_')[0]
        d = load_pub_dict(fp)
        if d:
            test_docs[pxd] = d

print(f'Loaded {len(test_docs)} test papers from {TEST_TEXT_DIR}')
for pxd, d in test_docs.items():
    print(f'  {pxd}: {len(get_text(d)):,} chars')

Loaded 16 test papers from c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\TestPubText
  PubText: 0 chars
  PXD004010: 11,237 chars
  PXD016436: 7,509 chars
  PXD019519: 45,042 chars
  PXD025663: 20,717 chars
  PXD040582: 19,714 chars
  PXD050621: 14,085 chars
  PXD061009: 34,629 chars
  PXD061090: 19,600 chars
  PXD061136: 15,870 chars
  PXD061195: 9,808 chars
  PXD061285: 34,103 chars
  PXD062014: 29,451 chars
  PXD062469: 16,510 chars
  PXD062877: 35,268 chars
  PXD064564: 19,510 chars


## 4. PRIDE API + ProteomeXchange XML

Each test PXD is a real registered dataset. PRIDE stores structured metadata
(organism, tissue, disease, instrument, label) that researchers entered at submission time.
This is cleaner than anything we can extract from prose.

In [9]:
http_session = requests.Session()
http_session.headers.update({'User-Agent': 'SDRF-Extractor/1.0'})

def fetch_pride(pxd):
    """Fetch project metadata from PRIDE Archive REST API v2."""
    try:
        r = http_session.get(
            f'https://www.ebi.ac.uk/pride/ws/archive/v2/projects/{pxd}',
            timeout=PRIDE_TIMEOUT
        )
        if r.status_code != 200:
            return {}
        d = r.json()
        out = defaultdict(list)

        # Organism
        for o in d.get('organisms', []):
            name = o.get('name', '')
            if name:
                norm = organism_norm.get(name.lower().strip())
                out['Characteristics[Organism]'].append(norm or name.strip())

        # Tissue / OrganismPart
        for op in (d.get('organisms_part') or d.get('tissues') or []):
            name = op.get('name', ''); acc = op.get('accession', '')
            if name and name.lower() not in ('not available', 'n/a', ''):
                val = f'NT={name};AC={acc}' if acc else f'NT={name}'
                out['Characteristics[OrganismPart]'].append(val)

        # Disease
        for dis in d.get('diseases', []):
            name = dis.get('name', '')
            if name and name.lower() not in ('not available', 'n/a', 'none', 'normal', ''):
                out['Characteristics[Disease]'].append(name)

        # Instrument
        for inst in d.get('instruments', []):
            name = inst.get('name', ''); acc = inst.get('accession', '')
            if name:
                fmt = fmt_instrument(name)
                out['Comment[Instrument]'].append(
                    fmt if 'AC=' in fmt else (f'AC={acc};NT={name}' if acc else name)
                )

        # Label / quantification
        for qm in d.get('quantification_methods', []):
            name = qm.get('name', '')
            if name:
                out['Characteristics[Label]'].append(fmt_label(name))

        return {k: list(dict.fromkeys(v)) for k, v in out.items() if v}
    except Exception as e:
        print(f'  PRIDE error {pxd}: {e}')
        return {}

def fetch_px_xml(pxd):
    """ProteomeXchange XML — backup instrument/organism source."""
    out = defaultdict(list)
    try:
        r = http_session.get(
            f'https://proteomecentral.proteomexchange.org/cgi/GetDataset'
            f'?ID={pxd}&outputMode=XML&test=no',
            timeout=PX_TIMEOUT
        )
        if r.status_code != 200:
            return {}
        xml = r.text
        for m in re.finditer(r'<cvParam[^>]+accession="(MS:\d+)"[^>]+name="([^"]+)"', xml):
            if 'instrument' in m.group(2).lower():
                out['Comment[Instrument]'].append(fmt_instrument(m.group(2)))
        for m in re.finditer(r'<cvParam[^>]+accession="(NEWT:\d+)"[^>]+name="([^"]+)"', xml):
            tax  = m.group(1).replace('NEWT:', '')
            name = m.group(2)
            norm = organism_norm.get(name.lower().strip())
            out['Characteristics[Organism]'].append(norm if norm else f'{tax} ({name})')
    except Exception as e:
        print(f'  PX XML error {pxd}: {e}')
    return {k: list(dict.fromkeys(v)) for k, v in out.items() if v}

# Quick smoke test
test_pxds = list(test_docs.keys())
print(f'Will query PRIDE for {len(test_pxds)} PXDs: {test_pxds[:5]} ...')

Will query PRIDE for 16 PXDs: ['PubText', 'PXD004010', 'PXD016436', 'PXD019519', 'PXD025663'] ...


## 5. Regex extraction

Extracts protocol parameters from paper text. Regex wins on structured fields
(mass tolerances, gradient time, cleavage agent) where answers follow predictable patterns.

In [10]:
_NEG = r'(?<!without\s)(?<!no\s)(?<!not\s)'
_CLINICAL = re.compile(
    r'\b(patient|cohort|biopsy|tumor|tumour|cancer|carcinoma|malignant|'
    r'diagnosed|clinical|disease|healthy\s+(?:control|donor)|specimen|'
    r'hospital|surgical|resection|case|control)\b', re.I
)

def regex_extract(pub_dict):
    text     = get_text(pub_dict)
    text_low = text.lower()
    out = defaultdict(list)

    def add(col, val):
        if val and val not in out[col]:
            out[col].append(val)

    # ── Organism ────────────────────────────────────────────────────────
    for pat, key in [
        (re.compile(r'\b(homo\s+sapiens|human(?:s)?)\b', re.I), 'human'),
        (re.compile(r'\b(mus\s+musculus|mouse|mice|murine)\b', re.I), 'mouse'),
        (re.compile(r'\b(rattus\s+norvegicus|rat(?:s)?)\b', re.I), 'rat'),
        (re.compile(r'\b(saccharomyces\s+cerevisiae|(?<!\w)yeast(?!\w))\b', re.I), 'yeast'),
        (re.compile(r'\b(escherichia\s+coli|e\.?\s*coli)\b', re.I), 'e. coli'),
        (re.compile(r'\b(drosophila\s+melanogaster|fruit\s+fly)\b', re.I), 'drosophila melanogaster'),
        (re.compile(r'\b(danio\s+rerio|zebrafish)\b', re.I), 'zebrafish'),
        (re.compile(r'\b(sus\s+scrofa|porcine|pig(?:s)?)\b', re.I), 'pig'),
        (re.compile(r'\b(bos\s+taurus|bovine)\b', re.I), 'bovine'),
        (re.compile(r'\b(gallus\s+gallus|chicken)\b', re.I), 'chicken'),
        (re.compile(r'\b(arabidopsis\s+thaliana)\b', re.I), 'arabidopsis thaliana'),
    ]:
        if pat.search(text):
            add('Characteristics[Organism]', organism_norm.get(key, key))

    # ── OrganismPart ─────────────────────────────────────────────────────
    for tissue, norm in tissue_norm.items():
        if re.search(r'\b' + re.escape(tissue) + r'\b', text_low):
            add('Characteristics[OrganismPart]', norm)

    # ── CellLine ──────────────────────────────────────────────────────────
    for cl in CELL_LINES:
        if re.search(r'\b' + re.escape(cl.lower()) + r'\b', text_low):
            add('Characteristics[CellLine]', cl)

    # ── CleavageAgent ─────────────────────────────────────────────────────
    for pat, val in [
        (re.compile(_NEG + r'\b(trypsin(?:/lys[\s\-]?c)?)\b', re.I), 'AC=MS:1001251;NT=Trypsin'),
        (re.compile(_NEG + r'\b(lys[\s\-]?c)\b', re.I), 'AC=MS:1001255;NT=Lys-C'),
        (re.compile(_NEG + r'\b(glu[\s\-]?c)\b', re.I), 'AC=MS:1001917;NT=Glu-C'),
        (re.compile(_NEG + r'\b(chymotrypsin)\b', re.I), 'AC=MS:1001306;NT=Chymotrypsin'),
        (re.compile(_NEG + r'\b(asp[\s\-]?n)\b', re.I), 'AC=MS:1001267;NT=Asp-N'),
    ]:
        if pat.search(text):
            add('Characteristics[CleavageAgent]', val); break

    # ── Label ─────────────────────────────────────────────────────────────
    for pat, fn in [
        (re.compile(r'\b(tmt[\s\-]?(?:pro)?[\s\-]?(?:10|11|16|18|6|2)?(?:plex)?)\b', re.I), fmt_label),
        (re.compile(r'\b(itraq[\s\-]?(?:4|8)?(?:plex)?)\b', re.I), fmt_label),
        (re.compile(r'\b(silac)\b', re.I), lambda _: 'AC=MS:1002791;NT=SILAC'),
        (re.compile(r'\b(label[\s\-]free|lfq)\b', re.I), lambda _: 'AC=MS:1002038;NT=label free sample'),
        (re.compile(r'\b(tandem\s+mass\s+tag)\b', re.I), lambda _: 'AC=MS:1002453;NT=TMT6plex'),
    ]:
        m = pat.search(text)
        if m:
            add('Characteristics[Label]', fn(m.group(1))); break

    # ── ReductionReagent ──────────────────────────────────────────────────
    for pat, val in [
        (re.compile(_NEG + r'\b(dtt|dithiothreitol)\b', re.I), 'AC=MS:1000578;NT=DTT'),
        (re.compile(_NEG + r'\b(tcep)\b', re.I), 'AC=MS:1001135;NT=TCEP'),
    ]:
        if pat.search(text):
            add('Characteristics[ReductionReagent]', val); break

    # ── AlkylationReagent ─────────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(iodoacetamide|iaa)\b', re.I), 'AC=PRIDE:0000126;NT=Iodoacetamide'),
        (re.compile(r'\b(n[\s\-]?ethylmaleimide|nem)\b', re.I), 'AC=PRIDE:0000459;NT=N-ethylmaleimide'),
        (re.compile(r'\b(chloroacetamide|caa)\b', re.I), 'AC=PRIDE:0000126;NT=Chloroacetamide'),
    ]:
        if pat.search(text):
            add('Characteristics[AlkylationReagent]', val); break

    # ── Modifications ─────────────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(carbamidomethyl(?:ation)?|iodoacetamide)\b', re.I),
         'NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed'),
        (re.compile(r'\b(oxidation(?:\s+of\s+methionine)?)\b', re.I),
         'NT=Oxidation;AC=UNIMOD:35;TA=M;MT=Variable'),
        (re.compile(r'\b(phospho(?:rylation)?)\b', re.I),
         'NT=Phospho;AC=UNIMOD:21;TA=S,T,Y;MT=Variable'),
        (re.compile(r'\b(acetyl(?:ation)?)\b', re.I),
         'NT=Acetyl;AC=UNIMOD:1;TA=K;MT=Variable'),
        (re.compile(r'\b(ubiquitin(?:ation)?|di[\s\-]?glycine|gg[\s\-]?remnant)\b', re.I),
         'NT=GlyGly;AC=UNIMOD:121;TA=K;MT=Variable'),
        (re.compile(r'\b(deamidation|deamidated)\b', re.I),
         'NT=Deamidated;AC=UNIMOD:7;TA=N,Q;MT=Variable'),
    ]:
        if pat.search(text):
            add('Characteristics[Modification]', val)

    # ── AcquisitionMethod ─────────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(dda|data[\s\-]dependent)\b', re.I), 'AC=MS:1003215;NT=DDA'),
        (re.compile(r'\b(dia|data[\s\-]independent|swath)\b', re.I), 'AC=MS:1003215;NT=DIA'),
        (re.compile(r'\b(prm|parallel\s+reaction\s+monitoring)\b', re.I), 'AC=MS:1001501;NT=PRM'),
    ]:
        if pat.search(text):
            add('Comment[AcquisitionMethod]', val); break

    # ── Instrument ────────────────────────────────────────────────────────
    for pat in [
        re.compile(r'\b(Q[\s\-]?Exactive[\s\-]?HF[\s\-]?X)\b', re.I),
        re.compile(r'\b(Q[\s\-]?Exactive[\s\-]?HF)\b', re.I),
        re.compile(r'\b(Q[\s\-]?Exactive[\s\-]?Plus)\b', re.I),
        re.compile(r'\b(Q[\s\-]?Exactive)\b', re.I),
        re.compile(r'\b(Orbitrap\s+Astral)\b', re.I),
        re.compile(r'\b(Orbitrap\s+Fusion\s+Lumos)\b', re.I),
        re.compile(r'\b(Orbitrap\s+Fusion)\b', re.I),
        re.compile(r'\b(Orbitrap\s+Eclipse)\b', re.I),
        re.compile(r'\b(Orbitrap\s+Exploris\s+480|Exploris\s+480)\b', re.I),
        re.compile(r'\b(LTQ[\s\-]?Orbitrap\s+Velos)\b', re.I),
        re.compile(r'\b(LTQ[\s\-]?Orbitrap\s+Elite)\b', re.I),
        re.compile(r'\b(LTQ[\s\-]?Orbitrap)\b', re.I),
        re.compile(r'\b(timsTOF\s+Pro)\b', re.I),
        re.compile(r'\b(timsTOF)\b', re.I),
        re.compile(r'\b(Triple[\s\-]?TOF\s+6600)\b', re.I),
        re.compile(r'\b(Triple[\s\-]?TOF)\b', re.I),
    ]:
        m = pat.search(text)
        if m:
            add('Comment[Instrument]', fmt_instrument(m.group(1))); break

    # ── FragmentationMethod ───────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(hcd)\b', re.I), 'AC=MS:1002481;NT=HCD'),
        (re.compile(r'\b(cid)\b', re.I), 'AC=MS:1001880;NT=CID'),
        (re.compile(r'\b(etd)\b', re.I), 'AC=MS:1001526;NT=ETD'),
    ]:
        if pat.search(text):
            add('Comment[FragmentationMethod]', val)

    # ── IonizationType ────────────────────────────────────────────────────
    if re.search(r'\b(nano[\s\-]?esi|nesi)\b', text, re.I):
        add('Comment[IonizationType]', 'AC=MS:1000398;NT=nanoESI')
    elif re.search(r'\b(electrospray|esi)\b', text, re.I):
        add('Comment[IonizationType]', 'AC=MS:1000073;NT=ESI')

    # ── MS2MassAnalyzer ───────────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(orbitrap)\b', re.I), 'AC=MS:1000484;NT=Orbitrap'),
        (re.compile(r'\b(ion\s*trap)\b', re.I), 'AC=MS:1000264;NT=ion trap'),
        (re.compile(r'\b(tof)\b', re.I), 'AC=MS:1000084;NT=TOF'),
    ]:
        if pat.search(text):
            add('Comment[MS2MassAnalyzer]', val); break

    # ── FractionationMethod ───────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(scx|strong\s+cation\s+exchange)\b', re.I), 'AC=PRIDE:0000228;NT=SCX'),
        (re.compile(r'\b(hprp|high[\s\-]?ph\s+(?:rp|reversed[\s\-]phase))\b', re.I),
         'AC=PRIDE:0000550;NT=High-pH Reversed-Phase'),
        (re.compile(r'\b(sds[\s\-]?page)\b', re.I), 'AC=PRIDE:0000672;NT=SDS-PAGE'),
    ]:
        if pat.search(text):
            add('Comment[FractionationMethod]', val); break

    # ── Separation ────────────────────────────────────────────────────────
    if re.search(r'\b(nano[\s\-]?lc)\b', text, re.I):
        add('Comment[Separation]', 'AC=PRIDE:0000565;NT=nanoLC')
    elif re.search(r'\b(uplc|uhplc)\b', text, re.I):
        add('Comment[Separation]', 'UPLC')

    # ── Sex ───────────────────────────────────────────────────────────────
    if re.search(r'\b(male\s+and\s+female|both\s+sexes)\b', text, re.I):
        add('Characteristics[Sex]', 'male and female')
    elif re.search(r'\b(male\s+(?:donors?|subjects?|patients?|mice|rats?))\b', text, re.I):
        add('Characteristics[Sex]', 'male')
    elif re.search(r'\b(female\s+(?:donors?|subjects?|patients?|mice|rats?))\b', text, re.I):
        add('Characteristics[Sex]', 'female')

    # ── Disease (only when clinical context detected) ─────────────────────
    if _CLINICAL.search(text):
        for pat, val in [
            (re.compile(r'\b(alzheimer[\s\']?s?\s+disease)\b', re.I), 'Alzheimer disease'),
            (re.compile(r'\b(parkinson[\s\']?s?\s+disease)\b', re.I), 'Parkinson disease'),
            (re.compile(r'\b(type\s+2\s+diabetes|t2d(?:m)?)\b', re.I), 'type 2 diabetes mellitus'),
            (re.compile(r'\b(breast\s+(?:cancer|carcinoma))\b', re.I), 'breast carcinoma'),
            (re.compile(r'\b(colorectal\s+(?:cancer|carcinoma)|colon\s+cancer)\b', re.I), 'colorectal carcinoma'),
            (re.compile(r'\b(lung\s+(?:cancer|carcinoma))\b', re.I), 'lung carcinoma'),
            (re.compile(r'\b(glioblastoma|gbm)\b', re.I), 'glioblastoma'),
            (re.compile(r'\b(melanoma)\b', re.I), 'melanoma'),
            (re.compile(r'\b(prostate\s+(?:cancer|carcinoma))\b', re.I), 'prostate carcinoma'),
            (re.compile(r'\b(covid[\s\-]?19|sars[\s\-]?cov[\s\-]?2)\b', re.I), 'COVID-19'),
            (re.compile(r'\b(multiple\s+myeloma)\b', re.I), 'multiple myeloma'),
            (re.compile(r'\b(hepatocellular\s+carcinoma|hcc)\b', re.I), 'hepatocellular carcinoma'),
            (re.compile(r'\b(pancreatic\s+(?:cancer|ductal\s+adenocarcinoma)|pdac)\b', re.I),
             'pancreatic ductal adenocarcinoma'),
            (re.compile(r'\b(healthy\s+(?:controls?|donors?|volunteers?))\b', re.I), 'normal'),
        ]:
            if pat.search(text):
                add('Characteristics[Disease]', val)

    # ── MaterialType inference ─────────────────────────────────────────────
    if 'Characteristics[CellLine]' in out:
        add('Characteristics[MaterialType]', 'cell line')
    elif re.search(r'\b(tissue(?:s)?(?!\s+culture)|biopsy|tumor|tumour)\b', text, re.I):
        add('Characteristics[MaterialType]', 'tissue')
    elif re.search(r'\b(plasma|serum|urine|csf|saliva|whole\s+blood)\b', text, re.I):
        add('Characteristics[MaterialType]', 'biofluid')

    # ── Strain ────────────────────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(C57BL/6J?)\b'), 'C57BL/6J'),
        (re.compile(r'\b(BALB/c)\b'), 'BALB/c'),
        (re.compile(r'\b(Sprague[\s\-]Dawley)\b', re.I), 'Sprague-Dawley'),
        (re.compile(r'\b(Wistar)\b', re.I), 'Wistar'),
    ]:
        m = pat.search(text)
        if m:
            add('Characteristics[Strain]', val); break

    # ── Genotype ──────────────────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(wild[\s\-]?type|wt(?:\s+cells?|\s+mice)?)\b', re.I), 'wild-type'),
        (re.compile(r'\b(knockout|knock[\s\-]out|ko(?:\s+cells?|\s+mice)?)\b', re.I), 'knockout'),
        (re.compile(r'\b(transgenic)\b', re.I), 'transgenic'),
    ]:
        if pat.search(text):
            add('Characteristics[Genotype]', val); break

    # ── DevelopmentalStage ────────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(adult(?:s)?)\b', re.I), 'adult'),
        (re.compile(r'\b(embryo(?:nic)?)\b', re.I), 'embryo'),
        (re.compile(r'\b(fetal|fetus|foetal)\b', re.I), 'fetal'),
        (re.compile(r'\b(neonatal|newborn)\b', re.I), 'neonatal'),
    ]:
        if pat.search(text):
            add('Characteristics[DevelopmentalStage]', val); break

    # ── Numeric protocol fields ───────────────────────────────────────────
    for pat in [
        re.compile(r'(\d+)[\s\-]min(?:ute)?\s+(?:gradient|linear\s+gradient)\b', re.I),
        re.compile(r'gradient\s+(?:of\s+)?(\d+)[\s\-]?min\b', re.I),
    ]:
        m = pat.search(text)
        if m:
            add('Comment[GradientTime]', f'{m.group(1)} min'); break

    m = re.search(r'(\d+(?:\.\d+)?)\s*(nl|nL|µl|µL|ul|uL)\s*/\s*min', text)
    if m:
        unit = 'nL' if m.group(2).lower() == 'nl' else 'µL'
        add('Comment[FlowRateChromatogram]', f'{m.group(1)} {unit}/min')

    for pat in [
        re.compile(r'(?:precursor|ms1)\s+(?:mass\s+)?tolerance(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(ppm|da)', re.I),
        re.compile(r'(\d+(?:\.\d+)?)\s*ppm\s+(?:for\s+)?(?:precursor|ms1)', re.I),
    ]:
        m = pat.search(text)
        if m:
            unit = m.group(2) if m.lastindex and m.lastindex >= 2 else 'ppm'
            add('Comment[PrecursorMassTolerance]', f'{m.group(1)} {unit}'); break

    for pat in [
        re.compile(r'(?:fragment|ms2)\s+(?:mass\s+)?tolerance(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(ppm|da|mda)', re.I),
        re.compile(r'(\d+(?:\.\d+)?)\s*(da|mda)\s+(?:for\s+)?(?:fragment|ms2)', re.I),
    ]:
        m = pat.search(text)
        if m:
            unit = m.group(2) if m.lastindex and m.lastindex >= 2 else 'Da'
            add('Comment[FragmentMassTolerance]', f'{m.group(1)} {unit}'); break

    for pat in [
        re.compile(r'(?:up\s+to\s+|allowing\s+(?:up\s+to\s+)?)(\d)\s+missed\s+cleavages?', re.I),
        re.compile(r'missed\s+cleavages?\s*[=:≤]\s*(\d)', re.I),
    ]:
        m = pat.search(text)
        if m:
            add('Comment[NumberOfMissedCleavages]', m.group(1)); break

    for pat in [
        re.compile(r'(\d+)\s+(?:independent\s+)?biological\s+replicates?', re.I),
        re.compile(r'performed\s+in\s+(triplicate|duplicate|quadruplicate)\b', re.I),
    ]:
        m = pat.search(text)
        if m:
            wm = {'triplicate':'3','duplicate':'2','quadruplicate':'4'}
            val = wm.get(m.group(1).lower(), m.group(1)) if m.lastindex else '3'
            add('Characteristics[NumberOfBiologicalReplicates]', val); break

    for pat in [
        re.compile(r'(?:fractionated\s+into|divided\s+into)\s+(\d+)\s+fractions?', re.I),
        re.compile(r'(\d+)\s+(?:scx|hprp|rp)?\s*fractions?\s+(?:were|of)', re.I),
    ]:
        m = pat.search(text)
        if m:
            add('Comment[NumberOfFractions]', m.group(1)); break

    for pat in [
        re.compile(r'cohort\s+of\s+(\d+)\s+(?:patients?|subjects?|individuals?)', re.I),
        re.compile(r'total\s+of\s+(\d+)\s+samples?', re.I),
    ]:
        m = pat.search(text)
        if m:
            add('Characteristics[NumberOfSamples]', m.group(1)); break

    # Cap multi-value columns at 4
    for col in ['Characteristics[OrganismPart]','Characteristics[CellLine]',
                'Characteristics[Modification]','Characteristics[Disease]']:
        if col in out:
            out[col] = out[col][:4]

    return dict(out)

print('Regex extractor defined.')

Regex extractor defined.


## 6. BioBERT QA layer

Uses `dmis-lab/biobert-large-cased-v1.1-squad` as an extractive QA model.
Instead of pattern matching, it asks questions about the paper and extracts answer spans.

**Why this beats regex for hard columns:** Disease and tissue descriptions often appear
in non-standard phrasing that regex patterns miss. BioBERT was pre-trained on PubMed
and fine-tuned on SQuAD — it reads biomedical text the way a human annotator would.

**Only applied to columns regex struggles with:** Disease, OrganismPart (when not
already found), CellType, DevelopmentalStage.

In [12]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

print(f'Loading BioBERT QA model from: {BIOBERT_MODEL}')
print('This may take 60s on first load ...')

qa_tokenizer = AutoTokenizer.from_pretrained(BIOBERT_MODEL)
qa_model     = AutoModelForQuestionAnswering.from_pretrained(BIOBERT_MODEL)
qa_model.eval()
if device == 0:
    qa_model = qa_model.cuda()
print('BioBERT QA ready.')


Loading BioBERT QA model from: dmis-lab/biobert-large-cased-v1.1-squad
This may take 60s on first load ...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 31002.55it/s]
BertForQuestionAnswering LOAD REPORT from: dmis-lab/biobert-large-cased-v1.1-squad
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BioBERT QA ready.


In [13]:
QA_QUESTIONS = [
    ('Characteristics[Disease]', [
        'What disease or condition is studied in this paper?',
        'What is the disease diagnosis of the patients?',
        'What pathology or disorder is investigated?',
    ]),
    ('Characteristics[OrganismPart]', [
        'What tissue or organ was the sample collected from?',
        'What biological fluid or tissue was analyzed?',
        'What is the sample source tissue?',
    ]),
    ('Characteristics[CellType]', [
        'What type of cells were used in this study?',
        'What primary cell type was analyzed?',
    ]),
    ('Characteristics[DevelopmentalStage]', [
        'What developmental stage or age group were the subjects?',
        'Were the subjects adult, embryo, fetal, or neonatal?',
    ]),
]

QA_MIN_SCORE = 0.15
QA_MAX_CHARS = 2000

def _run_qa(question, context):
    """Run a single QA query, return (answer_str, score)."""
    inputs = qa_tokenizer(
        question, context,
        return_tensors='pt',
        truncation=True, max_length=512,
        padding=True,
    )
    if device == 0:
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        outputs = qa_model(**inputs)
    start = torch.argmax(outputs.start_logits).item()
    end   = torch.argmax(outputs.end_logits).item()
    if end < start:
        end = start
    # Confidence: softmax probability of best span
    start_prob = torch.softmax(outputs.start_logits, dim=-1)[0, start].item()
    end_prob   = torch.softmax(outputs.end_logits,   dim=-1)[0, end].item()
    score      = (start_prob + end_prob) / 2.0
    tokens     = inputs['input_ids'][0][start: end + 1]
    answer     = qa_tokenizer.decode(tokens, skip_special_tokens=True).strip()
    return answer, score

def biobert_extract(pub_dict, existing_vals):
    """Run QA only on columns not already filled by PRIDE/regex."""
    text = get_text(pub_dict, max_chars=QA_MAX_CHARS)
    if not text.strip():
        return {}

    out = {}
    for col, questions in QA_QUESTIONS:
        if col in existing_vals and existing_vals[col]:
            continue

        best_answer = None
        best_score  = QA_MIN_SCORE

        for question in questions:
            try:
                answer, score = _run_qa(question, text)
                if score > best_score and len(answer) > 2:
                    best_score  = score
                    best_answer = answer
            except Exception:
                continue

        if best_answer:
            base_col = re.sub(r'\.\d+$', '', col)
            snapped  = fuzzy_snap(best_answer, base_col, cutoff=0.75)
            out[col] = [snapped]

    return out

print('BioBERT QA extractor defined.')
print(f'Columns targeted: {[q[0] for q in QA_QUESTIONS]}')


BioBERT QA extractor defined.
Columns targeted: ['Characteristics[Disease]', 'Characteristics[OrganismPart]', 'Characteristics[CellType]', 'Characteristics[DevelopmentalStage]']


## 7. Filename token parser

In [14]:
def parse_filename_tokens(raw_files):
    """Extract per-file metadata from raw filename patterns."""
    n = len(raw_files)
    results = {
        'Comment[FractionIdentifier]':         [None] * n,
        'Characteristics[BiologicalReplicate]': [None] * n,
        'Characteristics[TechnicalReplicate]':  [None] * n,
        'Characteristics[Label]':               [None] * n,
    }
    for i, rf in enumerate(raw_files):
        rf_str = str(rf); rf_up = rf_str.upper()
        m = re.search(r'(?:_f|_fr|_frac(?:tion)?)[_\s]?(\d+)', rf_str, re.I)
        if m:
            results['Comment[FractionIdentifier]'][i] = m.group(1)
        m = re.search(r'(?:_rep|_br|_biol?rep|_biorep|[_\-]r(?=\d))(\d+)', rf_str, re.I)
        if m:
            results['Characteristics[BiologicalReplicate]'][i] = f'biological replicate {m.group(1)}'
        m = re.search(r'(?:_tr|_tech(?:rep)?|_inj|_techrep)(\d+)', rf_str, re.I)
        if m:
            results['Characteristics[TechnicalReplicate]'][i] = f'technical replicate {m.group(1)}'
        if re.search(r'TMT', rf_up):
            mp = re.search(r'TMT(\d+)', rf_up)
            results['Characteristics[Label]'][i] = fmt_label(f'tmt{mp.group(1)}' if mp else 'tmt')
        elif re.search(r'SILAC|HEAVY|_H_|LIGHT|_L_', rf_up):
            results['Characteristics[Label]'][i] = 'AC=MS:1002791;NT=SILAC'
        elif re.search(r'LFQ|LF_|LABELFREE', rf_up):
            results['Characteristics[Label]'][i] = 'AC=MS:1002038;NT=label free sample'
    return results

print('Filename token parser defined.')

Filename token parser defined.


## 8. Main extraction pipeline

For each test PXD, runs all four layers in priority order:
1. PRIDE API (structured DB metadata)
2. ProteomeXchange XML (backup)
3. Regex (protocol params from paper text)
4. BioBERT QA (hard columns regex missed)
5. Global training majority fallback (anything still empty)

In [15]:
predicted_sets = defaultdict(dict)

for pxd, pub_dict in tqdm(test_docs.items(), desc='Extracting'):
    col_vals = defaultdict(list)

    def add(col, val):
        if not val or str(val).strip().lower() in ('not applicable','na','n/a',''): return
        base_col    = re.sub(r'\.\d+$', '', col)
        snapped_val = fuzzy_snap(str(val).strip(), base_col)
        if snapped_val not in col_vals[col]:
            col_vals[col].append(snapped_val)

    def add_list(col, vals):
        for v in (vals or []): add(col, v)

    # ── Layer 0: Training overlap (if this PXD was in training) ──────────
    if pxd in train_pxd_sdrf:
        for col, vals in train_pxd_sdrf[pxd].items():
            add_list(col, vals)

    # ── Layer 1: PRIDE API ────────────────────────────────────────────────
    pride_data = fetch_pride(pxd)
    for col, vals in pride_data.items():
        add_list(col, vals)
    time.sleep(0.3)  # be polite to PRIDE

    # ── Layer 2: ProteomeXchange XML (backup) ─────────────────────────────
    for col, vals in fetch_px_xml(pxd).items():
        add_list(col, vals)

    # ── Layer 3: Regex ────────────────────────────────────────────────────
    for col, vals in regex_extract(pub_dict).items():
        add_list(col, vals)

    # ── Layer 4: BioBERT QA (only for columns still empty) ────────────────
    existing_snapshot = {col: vals[:] for col, vals in col_vals.items()}
    for col, vals in biobert_extract(pub_dict, existing_snapshot).items():
        add_list(col, vals)

    # ── Layer 5: Global fallback for high-frequency columns ───────────────
    found_base = set(re.sub(r'\.\d+$', '', c) for c in col_vals.keys())
    for col in list(all_base - found_base):
        if non_na_ratio.get(col, 0.0) > 0.80:
            add(col, global_modes.get(col, 'Not Applicable'))

    # ── Assign modification slots (Modification, Modification.1, etc.) ────
    mods = list(dict.fromkeys(col_vals.pop('Characteristics[Modification]', [])))
    for i, mod in enumerate(mods):
        slot = 'Characteristics[Modification]' if i == 0 else f'Characteristics[Modification].{i}'
        col_vals[slot] = [mod]

    predicted_sets[pxd] = {col: list(dict.fromkeys(v)) for col, v in col_vals.items() if v}

    # Print summary for this PXD
    filled = len([c for c in predicted_sets[pxd] if predicted_sets[pxd][c]])
    has_biobert = any(c in predicted_sets[pxd] for c, _ in QA_QUESTIONS)
    print(f'  {pxd}: {filled} cols filled  PRIDE={bool(pride_data)}  BioBERT={has_biobert}')

print(f'\nDone — {len(predicted_sets)} PXDs extracted.')

Extracting:   6%|▋         | 1/16 [00:04<01:07,  4.53s/it]

  PubText: 37 cols filled  PRIDE=False  BioBERT=True


Extracting:  12%|█▎        | 2/16 [00:08<01:02,  4.49s/it]

  PXD004010: 37 cols filled  PRIDE=True  BioBERT=True


Extracting:  19%|█▉        | 3/16 [00:12<00:52,  4.07s/it]

...

  PXD062469: 42 cols filled  PRIDE=True  BioBERT=True


Extracting:  94%|█████████▍| 15/16 [00:57<00:03,  3.63s/it]

  PXD062877: 43 cols filled  PRIDE=True  BioBERT=True


Extracting: 100%|██████████| 16/16 [01:00<00:00,  3.76s/it]

  PXD064564: 39 cols filled  PRIDE=True  BioBERT=True

Done — 16 PXDs extracted.


## 9. Build and save submission CSV

In [16]:
def assign_cycling(n_rows, values):
    """Cycle values across rows — handles per-file assignments."""
    return [values[i % len(values)] for i in range(n_rows)]

final_sub = sample_sub.copy()
for col in target_cols:
    final_sub[col] = 'Not Applicable'

for pxd, pxd_df in final_sub.groupby('PXD'):
    idx       = pxd_df.index
    extr      = predicted_sets.get(pxd, {})
    raw_files = pxd_df['Raw Data File'].tolist()
    fn_tokens = parse_filename_tokens(raw_files)

    for col in target_cols:
        base_col = re.sub(r'\.\d+$', '', col)

        # Filename-derived per-row columns take priority
        if col in fn_tokens:
            per_row = fn_tokens[col]
            if any(v is not None for v in per_row):
                for i, row_idx in enumerate(idx):
                    final_sub.at[row_idx, col] = per_row[i] or 'Not Applicable'
                if col == 'Characteristics[Label]': continue
                continue

        vals = extr.get(col) or extr.get(base_col) or []
        vals = [v for v in vals if str(v).strip().lower() not in ('not applicable', '')]

        # Label fallback from filenames if nothing extracted
        if col == 'Characteristics[Label]' and not vals:
            fn_labels = set()
            for rf in raw_files:
                ru = str(rf).upper()
                if re.search(r'TMT', ru):
                    mp = re.search(r'TMT(\d+)', ru)
                    fn_labels.add(fmt_label(f'tmt{mp.group(1)}' if mp else 'tmt'))
                elif re.search(r'SILAC|HEAVY|LIGHT', ru):
                    fn_labels.add('AC=MS:1002791;NT=SILAC')
                elif re.search(r'LFQ|LF_|LF\d', ru):
                    fn_labels.add('AC=MS:1002038;NT=label free sample')
            vals = list(fn_labels)

        if vals:
            assigned = assign_cycling(len(idx), vals)
            for i, row_idx in enumerate(idx):
                final_sub.at[row_idx, col] = assigned[i]
        else:
            fb = global_modes.get(col, 'Not Applicable') \
                 if non_na_ratio.get(col, 0.0) > 0.80 else 'Not Applicable'
            for row_idx in idx:
                final_sub.at[row_idx, col] = fb

# Clean up any residual nulls
final_sub = final_sub.fillna('Not Applicable')
if 'Unnamed: 0' in final_sub.columns:
    final_sub = final_sub.drop(columns=['Unnamed: 0'])
for col in target_cols:
    mask = final_sub[col].astype(str).str.strip().isin(
        ['TextSpan','nan','None','[]','','null','Not applicable','not available']
    )
    final_sub.loc[mask, col] = 'Not Applicable'

final_sub.to_csv(OUT_PATH, index=False)
print(f'Saved → {OUT_PATH}')
print(f'Shape : {final_sub.shape}')

Saved → c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_pride_regex_biobert.csv
Shape : (1659, 81)


## 10. Fill-rate summary

In [17]:
label_cols = [c for c in final_sub.columns if c not in ('ID','PXD','Raw Data File','Usage')]
rows = []
for col in label_cols:
    n = (final_sub[col] != 'Not Applicable').sum()
    rows.append((col, n, round(n / len(final_sub) * 100, 1)))

rows.sort(key=lambda x: -x[1])
print(f'{"Column":<50} {"Filled":>7} {"Pct":>6}')
print('-' * 66)
for col, n, pct in rows:
    if n > 0:
        print(f'{col:<50} {n:>7} {pct:>5}%')

zero = sum(1 for _, n, _ in rows if n == 0)
print(f'\nColumns with any fill : {len(rows)-zero} / {len(rows)}')
print(f'Columns all-NA         : {zero}')
print(f'\nUpload {OUT_PATH.name} to Kaggle.')

Column                                              Filled    Pct
------------------------------------------------------------------
Characteristics[Bait]                                 1659 100.0%
Characteristics[BiologicalReplicate]                  1659 100.0%
Characteristics[CellPart]                             1659 100.0%
Characteristics[CleavageAgent]                        1659 100.0%
Characteristics[Compound]                             1659 100.0%
Characteristics[Depletion]                            1659 100.0%
Characteristics[Label]                                1659 100.0%
Characteristics[MaterialType]                         1659 100.0%
Characteristics[Modification]                         1659 100.0%
Characteristics[Modification].1                       1659 100.0%
Characteristics[Modification].2                       1659 100.0%
Characteristics[Modification].3                       1659 100.0%
Characteristics[Modification].4                       1659 100.0%
Character

In [20]:
# Use in-memory output if available; otherwise load the saved submission CSV.
df = final_sub.copy() if 'final_sub' in globals() else pd.read_csv(OUT_PATH)

print(df['Characteristics[Disease]'].value_counts(dropna=False).head(10))
print(df['Characteristics[Modification]'].value_counts(dropna=False).head(5))

Characteristics[Disease]
COVID-19                                   1376
Not Applicable                              135
Prostate carcinoma                           32
Lung cancer                                  30
Alpers-huttenlocher syndrome                 24
Mitochondrial metabolism disease             24
Alzheimer's disease                           9
Osteoarthritis                                6
multiple myeloma                              5
Gerstmann-straussler-scheinker syndrome       4
Name: count, dtype: int64
Characteristics[Modification]
NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed    1569
NT=Phospho;AC=UNIMOD:21;TA=S,T,Y;MT=Variable      90
Name: count, dtype: int64
